In [1]:
# imports
import pennylane as qml

In [ ]:
# configure problem here
items = {}
weight = 0

In [ ]:
def cost(z: str, items: dict, weight: int, lam: float) -> float:
    '''
    The cost function for knapsack problem. Outputs total value - penalty. This is the objective function to maximize

    Inputs
    z: a bit string determining which items to evaluate (ex. '1011' means include item 1, 3 and 4)
    items: a dictionary that has each items (value, weight)
    weight: the total weight constraint for the knapsack, W
    lam: lambda value for the penalty term. Must be tuned to correctly penalize infeasible answers
    '''
    n = len(z) # total number of items
    value = sum(items[i][0] * z[i] for i in range(n))
    total_weight = sum(items[i][1] * z[i] for i in range(n))
    penalty = lam * max(0, total_weight - weight) ** 2 # to ensure we heavily penalize weights that exceed our weight constraint W
    return value - penalty

def cost_hamiltonian():
    '''
    Converting our classical cost function into a cost hamiltonian to be used in the quantum circuit.

    Inputs
    '''
    pass

### Quantum Circuit

In [ ]:

def qaoa_layer(n: int, gamma: float, beta: float, cost_h):
    '''
    One layer of the qaoa circuit. One layer consists of one mixer and one cost operation.

    Inputs
    n: number of wires (or number of items)
    gamma: angle for cost hamiltonian
    beta: angle for mixer operation
    cost_h: cost hamiltonian
    '''
    qml.ApproxTimeEvolution(cost_h, gamma, 1) # apply cost hamiltonian with gamma
    for i in range(n):
        qml.RX(2 * beta, wires=i) # mixer operation
        
# @qml.qnode(dev)
def qaoa_circuit(n: int, gammas: list, betas: list, p: int, cost_h):
    '''
    Full qaoa circuit, consisting of p layers (above function)

    Inputs
    n: number of wires (items)
    gammas: list of each gamma value for every cost operation (length = p)
    betas: list of each beta value for every mixer operation (length = p)
    p: number of layers that are going to be applied in the circuit
    cost_h: cost hamiltonian
    '''
    for i in range (n):
        qml.Hadamard(wires=i)
    for j in range(p):
        qaoa_layer(n, gammas[j], betas[j], cost_h)
    return qml.probs(wires=range(n))

### Optimization
The full qaoa workflow requires classical optimization to be effective. QAOA will work dependent on having a good p (layer depth), and the angles for each layer. So we must introduce classical optimization algorithms to determine these values.